## needs

layer:
- layer type
- layer depth

if hierarchical layer:
hierarchy_object: (list of them)
    - characteristics list
    - position
    - kde
    - cluster_points
    - next layer object

if exploration layer:
    - points


    

In [1]:
import numpy as np
import optuna
import pandas as pd
from kDBCV.DBCV import DBCV_score
from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.preprocessing import StandardScaler

import src.analysis.analysis_routine as ar
from src.analysis.clustering import compute_clusters
from src.analysis.dim_reducer import reduce_dimensionality
from src.types import Config
from src.ui.data import load_dataset
from src.ui.state import init_state

df = load_dataset()

config: Config = init_state()


2026-06-28 13:02:06.053 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-28 13:02:06.054 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-28 13:02:06.054 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-28 13:02:06.055 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-28 13:02:06.055 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-28 13:02:06.056 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-28 13:02:06.056 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-28 13:02:06.056 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-

In [4]:
X = df.to_numpy()
X = StandardScaler().fit_transform(X)

n_dims = X.shape[1]

In [5]:

def objective(trial):
    cfg: Config = config.copy()
    cfg["umap_n_neighbors"] = trial.suggest_int("umap_n_neighbors", 5, 50)
    cfg["umap_min_dist"] = trial.suggest_float("umap_min_dist", 0.0, 0.5)
    cfg["hclust_min_cluster_size"] = trial.suggest_int("hclust_min_cluster_size", 5, 50)
    cfg["hclust_min_samples"] = trial.suggest_int("hclust_min_samples", 1, 25)
    cfg["hclust_umap_n_components"] = trial.suggest_int("hclust_umap_n_components", n_dims//2, n_dims)

    X_reduced = reduce_dimensionality(method="UMAP", X=X, n_components=cfg["hclust_umap_n_components"], config=cfg)
    labels, _ = compute_clusters(X_reduced, method="HDBSCAN", config=cfg)

    # guard degenerate solutions
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_frac = np.mean(labels == -1)
    if n_clusters < 2 or noise_frac > 0.5:
        return -1.0  # DBCV is in [-1, 1]; worst possible

    score, _ = DBCV_score(X, labels)  # returns (score, None); scored in ORIGINAL space
    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)
print(study.best_params, study.best_value)

[I 2026-06-28 13:05:17,508] A new study created in memory with name: no-name-eec22d5f-b2cb-4d34-8024-22a5c718faf2
[I 2026-06-28 13:05:23,425] Trial 0 finished with value: -0.34205026993406373 and parameters: {'umap_n_neighbors': 23, 'umap_min_dist': 0.3759309261547088, 'hclust_min_cluster_size': 12, 'hclust_min_samples': 17, 'hclust_umap_n_components': 10}. Best is trial 0 with value: -0.34205026993406373.
[I 2026-06-28 13:05:29,482] Trial 1 finished with value: -0.3429043367185683 and parameters: {'umap_n_neighbors': 37, 'umap_min_dist': 0.43895691232106004, 'hclust_min_cluster_size': 42, 'hclust_min_samples': 13, 'hclust_umap_n_components': 7}. Best is trial 0 with value: -0.34205026993406373.
[I 2026-06-28 13:05:35,537] Trial 2 finished with value: -0.3429043367185683 and parameters: {'umap_n_neighbors': 37, 'umap_min_dist': 0.18638339296791967, 'hclust_min_cluster_size': 47, 'hclust_min_samples': 18, 'hclust_umap_n_components': 7}. Best is trial 0 with value: -0.34205026993406373.


{'umap_n_neighbors': 13, 'umap_min_dist': 0.15714736374605853, 'hclust_min_cluster_size': 5, 'hclust_min_samples': 4, 'hclust_umap_n_components': 7} -0.09133163218458777
